
# 🧭 CMPE 272 – In‑Class ERP Lab (Colab)  
**Topic:** ERP Systems — Order‑to‑Cash, Procure‑to‑Pay, Basic MRP, Data Quality, and Controls  
**Fictional Company:** **BlueFin eFoils, Inc.** 🏄‍♀️🔋  
**Team Size:** 2–4 students • **Time:** ~90–120 minutes

> **How to use this notebook**
> 1. Go to **Runtime → Run all** to install packages and generate the sample dataset.  
> 2. Complete every **`TODO`** you see (code and short written responses).  
> 3. Work as a team; rotate a *driver* (keyboard) every section.  
> 4. At the end, make sure your team’s names are in the header below and upload/submit as directed by your instructor.

**Team members:** `<Add names here>`

---

### Learning goals
- See how core ERP processes tie together: **Sales/Distribution**, **Inventory**, **Procurement**, **Production (MRP)**, and **Finance**.  
- Implement small but realistic ERP routines: allocations, reorders, and simple BOM planning.  
- Practice **data quality** cleanup and **roles/controls** reasoning.  
- Communicate decisions succinctly with evidence from your data.




## 0) Quick setup — run this first
This cell installs what we need and generates a small, messy-but-fun demo dataset. If you re-run, it will reset to the original state.


In [2]:

# @title Install & Imports (Run me)
# Minimal deps so the whole class can run quickly.
import pandas as pd
import numpy as np
import datetime as dt
import math
import textwrap
from collections import defaultdict

# Optional plotting (use only if you want): matplotlib is available by default in Colab.
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Random seed for reproducibility across teams
rng = np.random.default_rng(272)

# -------------------------------
# Scenario: BlueFin eFoils, Inc.
# -------------------------------
# BlueFin manufactures and sells electric hydrofoil surfboards (eFoils).
# Products:
#  - BF-100 Board Kit (finished good)
#  - BAT-9 Battery Pack
#  - CTRL-2 Control Unit
#  - FOIL-3 Hydrofoil Assembly
#  - CHR-1 Charger
# The BF-100 is assembled from BAT-9, CTRL-2, FOIL-3, and CHR-1.
# Sales spike is expected next month due to a viral video. Inventory is tight.

today = pd.Timestamp.today().normalize()

products = pd.DataFrame([
    {"sku":"BF-100","name":"BlueFin eFoil Board Kit","uom":"ea","unitCost":1800.00,"unitPrice":2999.00,"leadTimeDays":7,"reorderPoint":12,"reorderQty":20},
    {"sku":"BAT-9","name":"Battery Pack 9kWh","uom":"ea","unitCost":750.00,"unitPrice":999.00,"leadTimeDays":10,"reorderPoint":25,"reorderQty":40},
    {"sku":"CTRL-2","name":"Control Unit Mk2","uom":"ea","unitCost":220.00,"unitPrice":399.00,"leadTimeDays":8,"reorderPoint":25,"reorderQty":40},
    {"sku":"FOIL-3","name":"Hydrofoil Assembly v3","uom":"ea","unitCost":380.00,"unitPrice":599.00,"leadTimeDays":9,"reorderPoint":25,"reorderQty":40},
    {"sku":"CHR-1","name":"Fast Charger 2kW","uom":"ea","unitCost":95.00,"unitPrice":179.00,"leadTimeDays":5,"reorderPoint":30,"reorderQty":60},
])

inventory = pd.DataFrame([
    {"sku":"BF-100","warehouse":"SJ-01","onHand":14,"reserved":0,"inTransit":0},
    {"sku":"BAT-9","warehouse":"SJ-01","onHand":28,"reserved":0,"inTransit":10},
    {"sku":"CTRL-2","warehouse":"SJ-01","onHand":19,"reserved":0,"inTransit":0},
    {"sku":"FOIL-3","warehouse":"SJ-01","onHand":26,"reserved":0,"inTransit":0},
    {"sku":"CHR-1","warehouse":"SJ-01","onHand":32,"reserved":0,"inTransit":0},
])

# Simple BOM (one level for speed)
bom = pd.DataFrame([
    {"parent":"BF-100","component":"BAT-9","qtyPer":1},
    {"parent":"BF-100","component":"CTRL-2","qtyPer":1},
    {"parent":"BF-100","component":"FOIL-3","qtyPer":1},
    {"parent":"BF-100","component":"CHR-1","qtyPer":1},
])

# Customers
customers = pd.DataFrame([
    {"customerId":"C100","name":"Sunny Surf Co.","terms":"NET30"},
    {"customerId":"C200","name":"Wave Riders LLC","terms":"NET30"},
    {"customerId":"C300","name":"Direct-to-Consumer","terms":"CC"},
])

# Sample orders for the next two weeks (some will exceed stock)
orderLines = pd.DataFrame([
    {"orderId":"SO-001","orderDate":today + pd.Timedelta(days=1),"customerId":"C100","sku":"BF-100","qty":8},
    {"orderId":"SO-002","orderDate":today + pd.Timedelta(days=2),"customerId":"C200","sku":"BF-100","qty":7},
    {"orderId":"SO-003","orderDate":today + pd.Timedelta(days=2),"customerId":"C300","sku":"BF-100","qty":6},
    {"orderId":"SO-004","orderDate":today + pd.Timedelta(days=3),"customerId":"C100","sku":"BF-100","qty":9},
])

# Vendors (with messy/duplicate data on purpose)
vendors = pd.DataFrame([
    {"vendorId":"V01","name":"Voltix Power","phone":"(408) 555-0100","email":"orders@voltix.com","defaultSku":"BAT-9"},
    {"vendorId":"V01X","name":"Voltix  Power ","phone":"408-555-0100","email":"sales@voltix.com","defaultSku":"BAT-9"},
    {"vendorId":"V02","name":"FoilWorks","phone":"+1 408.555.0200","email":"contact@foilworks.io","defaultSku":"FOIL-3"},
    {"vendorId":"V03","name":"Controllytics","phone":"4085550300","email":"hello@controllytics.ai","defaultSku":"CTRL-2"},
    {"vendorId":"V04","name":"ChargeFast","phone":"(408)-555-0400","email":"orders@chargefast.co","defaultSku":"CHR-1"},
])

leadTimes = dict(zip(products["sku"], products["leadTimeDays"]))

# Simple GL (very lightweight for the exercise)
glAccounts = pd.DataFrame([
    {"account":"1200","name":"Accounts Receivable"},
    {"account":"1400","name":"Inventory"},
    {"account":"2000","name":"Accounts Payable"},
    {"account":"4000","name":"Revenue"},
    {"account":"5000","name":"COGS"},
])

print("✅ Dataset loaded. Try: products.head(), inventory, orderLines, vendors, bom")


✅ Dataset loaded. Try: products.head(), inventory, orderLines, vendors, bom



## 1) Company overview & quick process mapping (10–15 min)

**BlueFin eFoils, Inc.** designs and manufactures electric hydrofoil surfboards. They sell direct and to specialty surf dealers. A viral video just boosted demand for the **BF‑100** finished good, and the team is worried about **stockouts** and **component shortages**.

**Your task:** In teams, make a lean mapping of the two big flows:

- **Order‑to‑Cash (O2C):** From Sales Order → Allocation → Pick/Pack/Ship → Invoice → Cash Receipt → GL postings.  
- **Procure‑to‑Pay (P2P):** From Requisition → PO → Receiving → AP Invoice → Payment → GL postings.

### ✅ TODO (write below this cell)
1. Identify the **ERP modules** involved in O2C and P2P and list **key master data** used by each (customers, vendors, items, price lists, etc.).  
2. Write **2–3 key risks** for each process (e.g., wrong shipments, duplicate payments, etc.) and a **control** that mitigates each risk.
3. Sketch a simple text map of O2C and P2P (you can use arrows like `->`).

> *Tip:* Keep it short but specific. You’ll refer back to this in later steps.


1. ERP modules
O2C: 
Sales Order Management - customers, price lists
Allocation - items, locations, stock levels
Pick/Pack/Ship - shipping addresses, carriers
Invoice/Cash Receipt - payment terms
General Ledger(GL) postings - chart of accounts
P2P: 
From Requisition/Purchase Order(PO) - approved vendors/suppliers
Receiving - items, locations, units of measure
Acounts Payable(AP) Invoice/Payment - vendor terms, invoices, payment information
GL postings - chart of accounts

2. Risks/control
O2C:
Wrong shipments - barcode on the product for unique serial number
stockout - Database tracking item, type and stock number. Real time inventory checks that can update the database. 
P2P:
Incorrect receipts - receiving quantity/item verification
Unauthorized purchases - have an approved vendor and PO approval system

3. Process Map
Order‑to‑Cash (O2C): From Sales Order → Allocation → Pick/Pack/Ship → Invoice → Cash Receipt → GL postings.  
Procure‑to‑Pay (P2P): From Requisition → Purchase Order(PO) → Receiving → Acounts Payable(AP) Invoice → Payment → General Ledger(GL) postings.


## 2) Mini-sim: Order‑to‑Cash – allocate and invoice (20–25 min)

You’ll implement a tiny allocation & invoicing routine for **BF‑100** using the current `inventory` and `orderLines`.

**Business rules:**
- Allocate finished goods first‑come‑first‑served by `orderDate`.
- If you can’t fully allocate, record the **backorder** quantity.
- When allocated, create a simple **invoice** with `unitPrice` from `products` and compute line `amount`.

Below is a scaffold with `TODO` markers.


In [3]:

# Helper: get product price and cost quickly
def getProductRecord(sku: str) -> pd.Series:
    rec = products.loc[products["sku"] == sku]
    return rec.iloc[0] if not rec.empty else None

def allocateInventory(orderLinesDf: pd.DataFrame, inventoryDf: pd.DataFrame) -> pd.DataFrame:
    """
    Allocate on-hand inventory to orders by orderDate (FIFO).
    Returns a DataFrame with columns:
      orderId, orderDate, customerId, sku, qty, allocatedQty, backorderedQty
    """
    # TODO: Implement allocation logic
    # Steps:
    # 1) Sort orders by orderDate, then orderId (stable)
    # 2) Track available on-hand by sku (single warehouse for simplicity)
    # 3) For each order line, allocate up to available; compute backordered remainder
    # 4) Update the available bucket
    # 5) Return a DataFrame with allocated/backordered columns filled

    sortedLines = orderLinesDf.sort_values(["orderDate", "orderId"]).copy()
    sortedLines["allocatedQty"] = 0
    sortedLines["backorderedQty"] = sortedLines["qty"]  # default until allocated

    # >>> YOUR CODE HERE <<<
    # 1)
    orderLinesDf.sort_values(["orderDate", "orderId"], inplace=True, kind="stable") # inplace sorts the dataframe without creating a new one
    # 2)
    availableInventory = inventoryDf.set_index("sku")["onHand"].to_dict()
    # 3)
    for idx, line in orderLinesDf.iterrows():
        sku = line["sku"]
        qty = line["qty"]
        availableQty = availableInventory.get(sku, 0) # get the available quantity for the sku, default to 0 if not found
        allocatedQty = min(qty, availableQty)  # allocate as much as possible but not more than available
        backorderedQty = qty - allocatedQty # compute the backordered quantity as requested - allocated
        orderLinesDf.at[idx, "allocatedQty"] = allocatedQty
        orderLinesDf.at[idx, "backorderedQty"] = backorderedQty
        # 4)
        availableInventory[sku] = availableQty - allocatedQty 
    
    # 5)
    return sortedLines

def createInvoices(allocationsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Create invoices for lines with allocatedQty > 0.
    Columns: invoiceId, orderId, customerId, sku, allocatedQty, unitPrice, amount, invoiceDate, terms
    """
    # TODO:
    # 1) Filter allocated lines
    # 2) Lookup price and customer terms
    # 3) Compute amount = allocatedQty * unitPrice
    # 4) Generate invoiceId like INV-001, INV-002,...
    invRows = []

    # >>> YOUR CODE HERE <<<
    # 1)
    allocatedLines = allocationsDf[allocationsDf["allocatedQty"] > 0].copy()
    # 2)
    for idx, line in allocatedLines.iterrows():
        sku = line["sku"]
        unitPrice = getProductRecord(sku)["unitPrice"] # Lookup price
        customerId = line["customerId"] # Lookup customer term
        # 3)
        allocatedQty = line["allocatedQty"]
        amount = allocatedQty * unitPrice
        invoiceDate = dt.datetime.now().date()
        terms = customers.loc[customers["customerId"] == customerId, "terms"].values[0]
        invoiceId = f"INV-{idx+1:03d}"  # simple invoice ID generation
        invRows.append([invoiceId, line["orderId"], customerId, sku, allocatedQty, unitPrice, amount, invoiceDate, terms])

    invoicesDf = pd.DataFrame(invRows, columns=["invoiceId","orderId","customerId","sku","allocatedQty","unitPrice","amount","invoiceDate","terms"])
    return invoicesDf

# --- Quick test harness (you can run after implementing) ---
allocations = allocateInventory(orderLines, inventory)
invoices = createInvoices(allocations)
invoices.head()


,invoiceId,orderId,customerId,sku,allocatedQty,unitPrice,amount,invoiceDate,terms



## 3) Procure‑to‑Pay – smart reorders & receipts (20–25 min)

We’ll add a small **reorder suggestion** that generates POs when stock is below `reorderPoint`. Then we’ll **receive** those POs to update inventory.

**Business rules:**
- For each `sku`, compute **available** = `onHand + inTransit - reserved`.
- If `available < reorderPoint`, propose a PO for `reorderQty`.
- On receiving a PO, move qty from `inTransit` to `onHand`.

Then consider: *How would you reflect AP and cash movement in GL at a high level?* (Write short notes in markdown after the code.)

Below is a scaffold.


In [4]:

def suggestReorders(inventoryDf: pd.DataFrame, productsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame of suggested POs with columns:
      poId, sku, qty, etaDate, vendorGuess
    Vendor is guessed from `vendors.defaultSku` (simplified!)
    """
    # TODO:
    # 1) For each sku, compute available
    # 2) If below reorderPoint, propose reorderQty and ETA today + leadTimeDays
    # 3) Guess vendor by matching vendors.defaultSku == sku (first hit)
    poRows = []
    poCounter = 1

    # >>> YOUR CODE HERE <<<

    return pd.DataFrame(poRows, columns=["poId","sku","qty","etaDate","vendorGuess"])

def receivePo(inventoryDf: pd.DataFrame, poDf: pd.DataFrame) -> pd.DataFrame:
    """
    Simulate receiving all PO lines today.
    Moves qty from inTransit to onHand for matching sku at SJ-01.
    Returns an updated copy of inventoryDf.
    """
    inv = inventoryDf.copy()
    # TODO: implement simple receipt logic
    # >>> YOUR CODE HERE <<<
    return inv

# --- Quick test harness (run after you implement) ---
# pos = suggestReorders(inventory, products)
# inventory_after = receivePo(inventory, pos)
# inventory_after



## 4) Tiny MRP: explode BOM and plan components (15–20 min)

A dealer asked for **N units of BF‑100** next week. You’ll compute **gross requirements** for components and suggest **planned order releases** by lead time.

**Business rules (one-level BOM for speed):**
- Gross requirements for each component = demand for BF‑100 × `qtyPer` in `bom`.
- If component `available` (onHand + inTransit - reserved) is insufficient, suggest a planned release for the shortfall.
- Use `leadTimeDays` per `products` to set a **releaseDate** = `needDate - leadTimeDays` (assume needDate is next Friday).

Fill in the `TODO` below.


In [5]:

def planMrpForBf100(demandQty: int, needDate: pd.Timestamp) -> pd.DataFrame:
    """
    Returns a DataFrame with columns:
      component, grossReq, available, shortage, plannedReleaseQty, releaseDate
    """
    rows = []
    for _, r in bom[bom["parent"]=="BF-100"].iterrows():
        comp = r["component"]
        qtyPer = r["qtyPer"]
        gross = demandQty * qtyPer

        # Compute available
        invRow = inventory[(inventory["sku"]==comp) & (inventory["warehouse"]=="SJ-01")]
        available = int(invRow["onHand"].iloc[0] + invRow["inTransit"].iloc[0] - invRow["reserved"].iloc[0])

        # TODO: compute shortage and planned release by comparing to gross
        # >>> YOUR CODE HERE <<<
        shortage = None
        plannedReleaseQty = None

        # compute releaseDate = needDate - leadTimeDays
        lt = leadTimes.get(comp, 0)
        releaseDate = (needDate - pd.Timedelta(days=int(lt))).normalize()

        rows.append({
            "component": comp,
            "grossReq": gross,
            "available": available,
            "shortage": shortage,
            "plannedReleaseQty": plannedReleaseQty,
            "releaseDate": releaseDate,
        })
    return pd.DataFrame(rows)

# --- Quick test harness ---
# next_friday = (pd.Timestamp.today() + pd.offsets.Week(weekday=4)).normalize()
# mrp = planMrpForBf100(demandQty=15, needDate=next_friday)
# mrp



## 5) Data quality & governance (15–20 min)

The `vendors` master has **duplicates**, weird spacing, and **inconsistent phone formats**. Clean it!

**Success criteria:**
- Trim whitespace and normalize case where helpful.
- Standardize phone to E.164-lite: `+14085550100` style.
- Deduplicate records such that **"Voltix Power"** variants become a single vendor row (keep the most complete info).
- Explain your **dedupe key** and **merge logic** in a few bullets under the cell.

Fill in the `TODO` below.


In [6]:

def normalizePhone(phone: str) -> str:
    # Example normalizer to help you get started
    digits = "".join(ch for ch in str(phone) if ch.isdigit())
    if digits.startswith("1") and len(digits) == 11:
        return "+" + digits
    if len(digits) == 10:
        return "+1" + digits
    # fallback (you can improve this)
    return "+" + digits if digits else ""

def cleanVendorMaster(vendorsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Return a cleaned and de-duplicated vendor master.
    Hints:
      - .str.strip(), .str.title()
      - apply(normalizePhone)
      - groupby on a dedupe key you design
      - keep the "best" email/phone with the most characters
    """
    cleaned = vendorsDf.copy()

    # TODO: implement cleaning + dedupe
    # >>> YOUR CODE HERE <<<

    return cleaned

# --- Quick test harness ---
# vendors_clean = cleanVendorMaster(vendors)
# vendors_clean



## 6) Roles & controls: Who can do what? (10–15 min)

Suppose BlueFin uses these **roles** (SoD = Segregation of Duties):

- **SalesRep:** create orders, view inventory.
- **Warehouse:** allocate, ship goods.
- **Buyer:** create POs, receive goods.
- **APClerk:** enter vendor invoices, schedule payments.
- **Controller:** approve payments, post period close.

**Risks to avoid:** one person both **creates** and **pays** a vendor invoice; one person **creates** and **ships** a sales order.

**TODO:** Implement a simple access check and identify SoD conflicts.


In [7]:

roles = {
    "SalesRep": {"createSO","viewInv"},
    "Warehouse": {"allocate","ship","viewInv"},
    "Buyer": {"createPO","receive","viewInv"},
    "APClerk": {"enterAP","schedulePay"},
    "Controller": {"approvePay","postClose","viewInv"},
}

sodConflicts = [
    {"actions":{"enterAP","approvePay"}, "reason":"Same person could create and approve vendor payments."},
    {"actions":{"createSO","ship"}, "reason":"Same person could create and ship a fraudulent order."},
]

def canPerform(role: str, action: str) -> bool:
    """Return True if role includes the action."""
    # TODO: implement check
    # >>> YOUR CODE HERE <<<
    return False

def hasConflict(userRoles: set) -> list:
    """
    Given a set of roles assigned to a user, return a list of conflict reasons matched.
    A conflict matches if the union of actions across those roles covers all actions in a conflict rule.
    """
    # TODO: implement SoD conflict detection
    # >>> YOUR CODE HERE <<<
    return []

# --- Quick tests ---
# print(canPerform("SalesRep","viewInv"))   # expect True
# print(canPerform("SalesRep","ship"))      # expect False
# print(hasConflict({"APClerk","Controller"}))  # expect one conflict



## ⭐ Optional (if time): KPI mini‑plot (10 min)

Create a simple **daily fulfillment KPI**: for each day in `orderLines`, after running your allocation, compute how many units were **fulfilled** vs **backordered**. Plot a single chart with two lines.

**Hint:** Use `matplotlib` (no special styles/colors). Keep it simple.


In [8]:

# Example (fill in after you complete allocation):
# plt.figure()
# daily = allocations.groupby(allocations["orderDate"].dt.normalize())[["allocatedQty","backorderedQty"]].sum()
# daily.plot()
# plt.title("Daily BF-100: Allocated vs Backordered")
# plt.xlabel("Date")
# plt.ylabel("Units")
# plt.show()



## 7) Wrap‑up & quick reflection (5–10 min)
**As a team**, answer briefly:

1. Which **bottleneck** did you observe and what data would you collect next to improve it?  
2. If demand keeps spiking, which **policy change** (reorder points/qty, lead time buffers, safety stock) would you try first, and why?  
3. What **dashboard tile** (single KPI) would you pitch to leadership for next week? Include a one‑sentence rationale.



<details>
<summary><strong>Hints (expand if stuck)</strong></summary>

**Section 2 (Allocation):**
- Use a dict to track available stock by SKU: `available = {"BF-100": onHand}`.
- Allocation per line: `alloc = min(requested, available[sku])`; backorder is the rest.
- Don’t forget to subtract the allocation from `available`.

**Section 3 (Reorders):**
- Available = onHand + inTransit - reserved.
- Match vendor: `vendors.loc[vendors["defaultSku"]==sku].head(1)`.

**Section 4 (MRP):**
- Shortage = `max(0, grossReq - available)`; planned release = shortage (no lot sizing here).
- `releaseDate = needDate - pd.Timedelta(days=leadTimeDays)`.

**Section 5 (Data Quality):**
- Normalize name: `str.strip()`, squash multiple spaces: `" ".join(name.split())`.
- Choose longest non-empty email/phone when merging duplicates.

**Section 6 (Roles & Controls):**
- `canPerform` checks if `action in roles.get(role,set())`.
- For `hasConflict`, first union all actions across the user's roles, then check each conflict rule.

</details>



---

### ✅ What to submit
- This completed notebook with your **names at the top**.
- All **`TODO`** code sections implemented and short answers written.
- If you did the optional KPI, include the chart cell output.

> **Stretch idea:** Fork a copy and generalize the allocation to a multi‑SKU, multi‑warehouse scenario using the same approach.
